# Plano cartesiano, ponto e distância

**Módulo 07 — Geometria Analítica Plana e Espacial** · Notebook 01 de 22

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/07_geometria_analitica_plana_e_espacial/0701a_plano_cartesiano_ponto_e_distancia.ipynb)

**Pré-requisitos**
- Módulo 00 — Fundamentos e Python (todo): [`00_introducao_python.ipynb`](../00_fundamentos_e_python/00_introducao_python.ipynb)
- Módulo 01 — Conjuntos e Funções: [`0104a_relacoes.ipynb`](../01_conjuntos_e_funcoes/0104a_relacoes.ipynb) (par ordenado e plano cartesiano) e [`0110a_funcao_modular.ipynb`](../01_conjuntos_e_funcoes/0110a_funcao_modular.ipynb)
- Módulo 04 — Geometria Plana: [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) (ponto médio), [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) (classificação de triângulos), [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (pontos notáveis do triângulo), [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb) (Teorema de Tales e divisão de segmentos) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Pitágoras e sua recíproca)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- calcular a distância entre dois pontos quaisquer do plano e usá-la para classificar triângulos dados pelos vértices;
- encontrar o ponto médio de um segmento, o ponto que divide um segmento numa razão dada e o baricentro de um triângulo;
- obter o simétrico de um ponto em relação aos eixos, à origem e às bissetrizes dos quadrantes;
- explicar a diferença entre resolver um problema de geometria pela figura (método sintético) e pelas coordenadas (método analítico).

**Tempo estimado:** 80 a 100 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_ga.py`, o arquivo com as funções de desenho deste módulo.

Ele funciona como o `utils_3d.py` do módulo 06, com uma diferença importante: **neste módulo, quem faz as contas é você**. O `utils_ga.py` só desenha. As fórmulas, você vai programar aqui mesmo, nas células.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/07_geometria_analitica_plana_e_espacial/utils_ga.py"
    urllib.request.urlretrieve(url, "utils_ga.py")

import math
import utils_ga as uga
from ipywidgets import interact
import ipywidgets as widgets

## Como o jogo sabe que você foi visto?

Em muitos jogos, um guarda vigia uma área. Se você chegar perto demais, ele te vê e o alarme dispara. Parece simples: é só olhar para a tela e ver se você está dentro do círculo de visão dele.

Mas o computador **não olha para a tela**. Ele não enxerga círculos, bonecos nem cenários. Tudo o que ele tem são **números**: a posição do guarda é um par de números, a sua posição é outro par de números, e o alcance da visão é mais um número.

Então, como ele decide, sessenta vezes por segundo, se você foi visto?

🕹️ Mexa nos controles abaixo para mover o seu personagem (o ponto $J$). O guarda está no ponto $G$, e o círculo é o alcance da visão dele.

In [2]:
# Não espie! Você vai escrever esta função na seção 2.
def jogador_foi_visto(jogador: tuple, guarda: tuple, raio: float) -> bool:
    """Decide se o jogador está dentro do alcance da visão do guarda."""
    delta_x = jogador[0] - guarda[0]
    delta_y = jogador[1] - guarda[1]
    return delta_x**2 + delta_y**2 <= raio**2

In [3]:
GUARDA = (6, 4)   # a posição do guarda
ALCANCE = 3       # o alcance da visão do guarda


def mostrar_guarda(x_jogador: float, y_jogador: float) -> None:
    """Desenha o guarda, o alcance da visão dele e o jogador, e diz se ele foi visto."""
    jogador = (x_jogador, y_jogador)

    fig = uga.criar_plano_cartesiano(
        titulo="O guarda (G) e você (J)", x_min=-2, x_max=10, y_min=-2, y_max=10
    )
    fig = uga.desenhar_circulo_de_alcance(fig, GUARDA, ALCANCE)
    fig = uga.desenhar_ponto(fig, GUARDA, rotulo="G", cor=uga.CORES["destaque"])
    fig = uga.desenhar_ponto(fig, jogador, rotulo="J", mostrar_coordenadas=True)
    uga.mostrar(fig)

    if jogador_foi_visto(jogador, GUARDA, ALCANCE):
        print("🚨 Você foi visto!")
    else:
        print("🤫 Você está escondido.")


interact(
    mostrar_guarda,
    x_jogador=widgets.FloatSlider(
        min=-2, max=10, step=0.5, value=2, description="x do jogador",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    y_jogador=widgets.FloatSlider(
        min=-2, max=10, step=0.5, value=1, description="y do jogador",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='x do jogador', max=10.0, mi…

Para o computador, a pergunta "o jogador está dentro do círculo?" precisa virar uma **conta** feita só com os números $(x_J, y_J)$, $(x_G, y_G)$ e o raio. No fim da seção 2 você vai escrever essa conta.

E essa é a grande ideia deste módulo inteiro: **transformar geometria em contas**. Foi o que o filósofo e matemático René Descartes propôs no século XVII. Neste módulo, você vai ver que pontos, retas, circunferências, planos e esferas podem ser descritos por números e equações, e que muitos problemas difíceis de resolver "olhando a figura" ficam simples quando viram álgebra.

## O que você já sabe e o que muda agora

Você não está começando do zero. Muita coisa deste notebook já apareceu antes, só que **de outro jeito**:

| Já visto | Onde | O que muda aqui |
|---|---|---|
| Par ordenado, plano cartesiano, quadrantes | 📎 Módulo 01 — [`0104a_relacoes.ipynb`](../01_conjuntos_e_funcoes/0104a_relacoes.ipynb) | Nada de reaprender a marcar pontos. Antes, o plano era o lugar onde você **desenhava** relações e gráficos de funções. Agora ele é uma **ferramenta para resolver problemas de geometria** com contas. |
| A fórmula da distância, usada "de confiança" em `math.dist` e justificada por Pitágoras | 📎 Módulo 04 — [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (seção 4) | Lá, ela foi uma aplicação de Pitágoras entre muitas. Aqui, ela vira **a ferramenta central**: vamos programá-la, testá-la em todos os casos e usá-la para classificar triângulos, decidir se o guarda te viu e até provar teoremas. |
| Classificação de triângulos e recíproca de Pitágoras | 📎 Módulo 04 — [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) | Você já sabe classificar um triângulo pelos lados e pelos ângulos. Aqui, você vai fazer isso **sem ver o triângulo**, só com as coordenadas dos vértices. |
| Ponto médio e divisão de um segmento numa razão | 📎 Módulo 04 — [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) e [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb) | Você já calculava o ponto médio como média das coordenadas e dividia segmentos na razão $m : n$. Aqui, essas contas ganham a justificativa pelo Teorema de Tales e uma forma mais simples (a **fração $t$ do caminho**), que é a mesma usada para animar personagens de jogos. |
| Teorema de Tales | 📎 Módulo 04 — [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb) | Ele vai justificar a fórmula do ponto médio. Não vamos demonstrá-lo de novo. |
| Baricentro e a razão 2 : 1 | 📎 Módulo 04 — [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) | Você já provou que o baricentro divide cada mediana na razão 2 : 1 e lá mesmo chegou à média dos vértices. Agora essa propriedade é a **ferramenta**: refazemos a conta com o que você programar aqui, e o computador confere as três medianas de uma vez. |
| Coordenadas só para desenhar | 📎 Módulo 06 — [`0601a_introducao_geometria_de_posicao.ipynb`](../06_geometria_espacial/0601a_introducao_geometria_de_posicao.ipynb) | No módulo 06 os eixos ficavam escondidos, porque as coordenadas eram só um truque do desenho. Agora elas são **o assunto**: os eixos aparecem e as contas ficam à vista. |

**Duas maneiras de fazer geometria**

| Geometria sintética (módulos 04 e 06) | Geometria analítica (este módulo) |
|---|---|
| Raciocina sobre a **figura** | Raciocina sobre **números e equações** |
| Usa congruência, semelhança, teoremas | Usa coordenadas, fórmulas e álgebra |
| Cada problema pede uma ideia esperta | Um mesmo procedimento serve para muitos problemas |
| Ótima para entender **por que** algo é verdade | Ótima para **calcular**, e o computador consegue fazer |

Nenhuma é melhor que a outra: elas se completam. Na seção 6 você vai provar o mesmo teorema dos dois jeitos e comparar.

### Retomada em 30 segundos

Um ponto do plano é dado pelo par ordenado $P(x_P, y_P)$: $x_P$ é a **abscissa** e $y_P$ é a **ordenada**. Os eixos dividem o plano em quatro **quadrantes** (📎 [`0104a_relacoes.ipynb`](../01_conjuntos_e_funcoes/0104a_relacoes.ipynb)).

> 💡 **Dica.** Nas figuras deste módulo, passe o mouse sobre qualquer ponto: o gráfico mostra as coordenadas dele. Use isso para conferir as suas contas.

In [4]:
# 1. Criamos um plano cartesiano vazio (com eixos e grade)
fig = uga.criar_plano_cartesiano(titulo="Pontos no plano")

# 2. Marcamos alguns pontos, mostrando as coordenadas
fig = uga.desenhar_ponto(fig, (3, 2), rotulo="A", mostrar_coordenadas=True)
fig = uga.desenhar_ponto(fig, (-4, 1), rotulo="B", mostrar_coordenadas=True)
fig = uga.desenhar_ponto(fig, (-2, -3), rotulo="C", mostrar_coordenadas=True)

# 3. As linhas tracejadas mostram de onde vêm as coordenadas de A
fig = uga.desenhar_projecoes(fig, (3, 2))

uga.mostrar(fig)

## 1. Distância entre dois pontos

### Primeiro, o caso fácil

Se dois pontos estão na mesma **horizontal** (mesma ordenada), a distância entre eles é só a diferença entre as abscissas. Por exemplo, de $A(1, 3)$ até $B(6, 3)$ são $6 - 1 = 5$ unidades.

E se fizermos a conta "ao contrário", $1 - 6 = -5$? Distância não pode ser negativa. Por isso usamos o **módulo** (📎 [`0110a_funcao_modular.ipynb`](../01_conjuntos_e_funcoes/0110a_funcao_modular.ipynb)):

$$d_{AB} = |x_B - x_A| \quad \text{(mesma ordenada)} \qquad d_{AB} = |y_B - y_A| \quad \text{(mesma abscissa)}$$

### Agora, o caso geral

E se os pontos não estiverem nem na mesma horizontal nem na mesma vertical? O truque é **enxergar um triângulo retângulo escondido**: andando de $A$ na horizontal e depois na vertical, chegamos a $B$. Os catetos medem $|\Delta x|$ e $|\Delta y|$, e a distância $d_{AB}$ é a hipotenusa.

Você já fez essa conta na seção 4 de 📎 [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb), quando a fórmula que o código usava desde [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) foi finalmente justificada. Vale relembrá-la com calma, porque ela é a base de todo este módulo: **entre dois pontos quaisquer sempre existe esse triângulo**, com os catetos paralelos aos eixos. Então Pitágoras dá, de uma vez, uma fórmula que vale para todos os casos:

$$d_{AB}^2 = |\Delta x|^2 + |\Delta y|^2$$

Como o quadrado de um número nunca é negativo, $|\Delta x|^2 = (\Delta x)^2$, e o módulo pode sumir:

> 📌 **Distância entre dois pontos.** Para $A(x_A, y_A)$ e $B(x_B, y_B)$:
> $$d_{AB} = \sqrt{(x_B - x_A)^2 + (y_B - y_A)^2}$$

Repare que a fórmula não liga para a ordem: $(x_B - x_A)^2 = (x_A - x_B)^2$. A distância de $A$ até $B$ é igual à de $B$ até $A$, como devia ser. E ela também funciona quando os pontos estão na mesma horizontal ou vertical: um dos catetos vale zero e a fórmula vira o caso fácil.

🕹️ Mova os pontos abaixo e veja o triângulo retângulo escondido aparecer.

In [5]:
def controle(nome: str, valor: int, minimo: int = -5, maximo: int = 5) -> widgets.IntSlider:
    """Cria um slider de números inteiros, que só atualiza a figura ao soltar o botão."""
    return widgets.IntSlider(
        value=valor, min=minimo, max=maximo, step=1, description=nome,
        style={"description_width": "initial"}, continuous_update=False,
    )


def entre_parenteses(numero: int) -> str:
    """Escreve um número negativo entre parênteses, como (−3), para a conta ficar legível."""
    return f"(−{abs(numero)})" if numero < 0 else str(numero)


def mostrar_distancia(xA: int, yA: int, xB: int, yB: int) -> None:
    """Desenha A, B e o triângulo escondido entre eles, e mostra a conta da distância."""
    A = (xA, yA)
    B = (xB, yB)

    # A conta, passo a passo
    delta_x = xB - xA
    delta_y = yB - yA
    d = math.sqrt(delta_x**2 + delta_y**2)

    fig = uga.criar_plano_cartesiano(titulo="O triângulo retângulo escondido entre A e B")
    fig = uga.desenhar_segmento(fig, A, B, cor=uga.CORES["reta"])
    fig = uga.desenhar_triangulo_das_diferencas(fig, A, B)
    fig = uga.desenhar_pontos(fig, [A, B], rotulos=["A", "B"], mostrar_coordenadas=True)
    uga.mostrar(fig)

    # A mesma conta, com os números no lugar das letras
    soma = delta_x**2 + delta_y**2
    final = f"= {round(d)}" if math.isclose(d, round(d)) else f"≈ {d:.2f}"
    print(
        f"d = √(({str(xB).replace('-', '−')} − {entre_parenteses(xA)})² + "
        f"({str(yB).replace('-', '−')} − {entre_parenteses(yA)})²) "
        f"= √({delta_x**2} + {delta_y**2}) = √{soma} {final}"
    )


interact(
    mostrar_distancia,
    xA=controle("x_A", -3), yA=controle("y_A", -1),
    xB=controle("x_B", 3), yB=controle("y_B", 2),
);

interactive(children=(IntSlider(value=-3, continuous_update=False, description='x_A', max=5, min=-5, style=Sli…

> ⚠️ **Erro comum 1.** Trocar $(x_B - x_A)^2$ por $x_B^2 - x_A^2$. São coisas diferentes! Para $x_A = 1$ e $x_B = 4$: $(4 - 1)^2 = 9$, mas $4^2 - 1^2 = 15$.

> ⚠️ **Erro comum 2.** Errar o sinal com coordenadas negativas. Se $x_A = -3$ e $x_B = 3$, então $\Delta x = 3 - (-3) = 6$, e não $0$. Use parênteses sempre que substituir um número negativo.

> ⚠️ **Erro comum 3.** Esquecer a raiz quadrada no final. $(\Delta x)^2 + (\Delta y)^2$ é o **quadrado** da distância.

### A fórmula vira uma função

Na geometria analítica, uma fórmula é praticamente um programa pronto. Veja como a distância fica em Python: cada ponto é uma **tupla** `(x, y)`.

In [6]:
Ponto = tuple[float, float]   # um ponto do plano é uma tupla (x, y)


def distancia(A: Ponto, B: Ponto) -> float:
    """Distância entre os pontos A = (xA, yA) e B = (xB, yB)."""
    xA, yA = A
    xB, yB = B
    return math.sqrt((xB - xA)**2 + (yB - yA)**2)


print(distancia((1, 2), (4, 6)))      # 5.0
print(distancia((4, 6), (1, 2)))      # 5.0: a ordem não importa
print(distancia((-3, -1), (2, 11)))   # 13.0

# Uma tabela com um par de pontos em cada situação
PARES = [
    ("mesmo quadrante", (1, 2), (4, 6)),
    ("quadrantes opostos", (-2, -3), (4, 5)),
    ("mesma horizontal", (-4, 3), (5, 3)),
    ("mesma vertical", (2, -5), (2, 1)),
    ("um deles na origem", (0, 0), (5, -12)),
]
print()
print(f"{'situação':<20}{'A':>10}{'B':>10}{'Δx':>6}{'Δy':>6}{'d':>8}")
for situacao, A, B in PARES:
    delta_x = B[0] - A[0]
    delta_y = B[1] - A[1]
    texto_A, texto_B = str(A), str(B)   # os pontos como texto, para alinhar na tabela
    print(
        f"{situacao:<20}{texto_A:>10}{texto_B:>10}"
        f"{delta_x:>6}{delta_y:>6}{distancia(A, B):>8.2f}"
    )

5.0
5.0
13.0

situação                     A         B    Δx    Δy       d
mesmo quadrante         (1, 2)    (4, 6)     3     4    5.00
quadrantes opostos    (-2, -3)    (4, 5)     6     8   10.00
mesma horizontal       (-4, 3)    (5, 3)     9     0    9.00
mesma vertical         (2, -5)    (2, 1)     0     6    6.00
um deles na origem      (0, 0)  (5, -12)     5   -12   13.00


> 📎 **Conexão (e uma caixa que se abre).** O Python tem essa fórmula pronta: `math.dist(A, B)`. Você usa essa função desde [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb), no módulo 04, e ela faz exatamente a conta que você acabou de programar. É assim com muitas funções prontas: por trás delas há uma fórmula matemática que alguém deduziu. Neste módulo, você vai abrir várias dessas caixas, inclusive algumas que o `utils_3d.py` usava em segredo no módulo 06.

In [7]:
# Conferindo a nossa função com a do Python, nos cinco pares da tabela
for situacao, A, B in PARES:
    nossa = distancia(A, B)
    do_python = math.dist(A, B)
    resultado = "✅ iguais" if math.isclose(nossa, do_python) else "❌ diferentes"
    print(f"{situacao:<20} distancia = {nossa:7.4f}   math.dist = {do_python:7.4f}   {resultado}")

mesmo quadrante      distancia =  5.0000   math.dist =  5.0000   ✅ iguais
quadrantes opostos   distancia = 10.0000   math.dist = 10.0000   ✅ iguais
mesma horizontal     distancia =  9.0000   math.dist =  9.0000   ✅ iguais
mesma vertical       distancia =  6.0000   math.dist =  6.0000   ✅ iguais
um deles na origem   distancia = 13.0000   math.dist = 13.0000   ✅ iguais


**Pare e pense**

1. Sem fazer contas: qual é maior, a distância de $(0, 0)$ até $(3, 4)$ ou de $(0, 0)$ até $(-3, -4)$?
2. Dois pontos têm distância 0. O que você pode concluir sobre eles?

<details>
<summary>Ver respostas</summary>

1. São iguais (as duas valem 5). Os sinais desaparecem quando elevamos ao quadrado: os dois pontos estão à mesma distância da origem, em lados opostos.
2. São o mesmo ponto. A soma $(\Delta x)^2 + (\Delta y)^2$ só vale 0 se $\Delta x = 0$ **e** $\Delta y = 0$.
</details>

## 2. O que dá para fazer com a distância

### 2.1 Classificar um triângulo sem olhar para ele

No módulo 04 você aprendeu a classificar triângulos (📎 [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb)):
- **pelos lados:** equilátero, isósceles ou escaleno;
- **pelos ângulos**, usando a recíproca de Pitágoras: chamando de $a$ o **maior** lado,
  - $a^2 = b^2 + c^2$ → retângulo;
  - $a^2 < b^2 + c^2$ → acutângulo;
  - $a^2 > b^2 + c^2$ → obtusângulo.

A novidade: com a fórmula da distância, basta conhecer as **coordenadas dos vértices**. O computador classifica o triângulo sem "ver" o desenho.

> 💡 **Dica.** Para comparar lados e aplicar a recíproca de Pitágoras, você nem precisa tirar a raiz: trabalhe direto com os **quadrados** das distâncias. Além de dar menos trabalho, isso evita erros de arredondamento.

In [8]:
def distancia_ao_quadrado(A: Ponto, B: Ponto) -> float:
    """O quadrado da distância entre A e B (a conta da distância, sem a raiz)."""
    xA, yA = A
    xB, yB = B
    return (xB - xA)**2 + (yB - yA)**2


def classificar_triangulo(A: Ponto, B: Ponto, C: Ponto) -> str:
    """Classifica o triângulo ABC pelos lados e pelos ângulos, só com as coordenadas.

    Atenção: a função supõe que A, B e C não estão alinhados. Detectar pontos
    alinhados com uma conta é assunto do notebook 02.
    """
    # Os quadrados dos três lados, do menor para o maior
    q1, q2, q3 = sorted([
        distancia_ao_quadrado(A, B),
        distancia_ao_quadrado(B, C),
        distancia_ao_quadrado(C, A),
    ])

    # Pelos lados: lados iguais têm quadrados iguais
    if math.isclose(q1, q3):
        lados = "equilátero"
    elif math.isclose(q1, q2) or math.isclose(q2, q3):
        lados = "isósceles"
    else:
        lados = "escaleno"

    # Pelos ângulos: recíproca de Pitágoras com o maior lado (q3 é o quadrado dele)
    if math.isclose(q3, q1 + q2):
        angulos = "retângulo"
    elif q3 < q1 + q2:
        angulos = "acutângulo"
    else:
        angulos = "obtusângulo"

    return f"{lados} e {angulos}"


TRIANGULOS = [
    ((0, 0), (4, 0), (1, 3)),
    ((1, 1), (4, 5), (5, -2)),
    ((0, 0), (6, 0), (1, 1)),
]

for A, B, C in TRIANGULOS:
    quadrados = sorted([distancia_ao_quadrado(A, B), distancia_ao_quadrado(B, C),
                        distancia_ao_quadrado(C, A)])
    classificacao = classificar_triangulo(A, B, C)
    print(f"A = {A}, B = {B}, C = {C}: quadrados dos lados {quadrados} → {classificacao}")

    fig = uga.criar_plano_cartesiano(
        titulo=classificacao, x_min=-1, x_max=7, y_min=-3, y_max=6, altura=350
    )
    fig = uga.desenhar_poligono(fig, [A, B, C], rotulos=["A", "B", "C"])
    uga.mostrar(fig)

A = (0, 0), B = (4, 0), C = (1, 3): quadrados dos lados [10, 16, 18] → escaleno e acutângulo


A = (1, 1), B = (4, 5), C = (5, -2): quadrados dos lados [25, 25, 50] → isósceles e retângulo


A = (0, 0), B = (6, 0), C = (1, 1): quadrados dos lados [2, 26, 36] → escaleno e obtusângulo


### 2.2 De volta ao guarda do jogo

Agora você consegue escrever a conta que o jogo faz. O jogador $J$ foi visto se a distância até o guarda $G$ é **menor ou igual** ao alcance da visão:

$$d_{JG} \le \text{raio}$$

Os programadores de jogos ainda usam um truque. Como as duas distâncias são positivas, comparar $d_{JG} \le \text{raio}$ é o mesmo que comparar os **quadrados**: $d_{JG}^2 \le \text{raio}^2$. Assim o computador não precisa calcular a raiz quadrada, que é uma conta mais lenta. Quando um jogo verifica milhares de personagens por segundo, isso faz diferença.

Complete a função abaixo e rode a célula seguinte para testá-la. Depois, mude a posição do jogador e veja se ela concorda com o gancho do começo do notebook.

In [10]:
# Exercício guiado (seção 2.2): a conta do guarda do jogo.
def foi_visto(jogador: Ponto, guarda: Ponto, raio: float) -> bool:
    # Complete aqui: devolva True se o jogador estiver dentro do alcance
    # (distância menor ou igual ao raio) e False caso contrário.
    # Desafio extra: faça sem usar raiz quadrada!
    ...

In [11]:
# Verificação
casos_do_guarda = [((3, 2), False), ((4, 4), True), ((6, 7), True), ((9, 7), False)]
for jogador, esperado in casos_do_guarda:
    assert foi_visto(jogador, (6, 4), 3) == esperado, (
        f"Tente de novo, revise: com J{jogador}, G(6, 4) e raio 3, a resposta deveria ser "
        f"{esperado}; compare a distância (ou o quadrado dela) com o raio "
        "(seção 2.2 - De volta ao guarda do jogo)."
    )
print("Certo!")

AssertionError: Tente de novo, revise: com J(3, 2), G(6, 4) e raio 3, a resposta deveria ser False; compare a distância (ou o quadrado dela) com o raio (seção 2.2 - De volta ao guarda do jogo).

**Pare e pense**

Por que comparar $d^2 \le r^2$ dá o mesmo resultado que comparar $d \le r$? Isso continuaria valendo se os números pudessem ser negativos?

<details>
<summary>Ver resposta</summary>

Para números **não negativos**, elevar ao quadrado preserva a ordem: se $0 \le d \le r$, então $d^2 \le r^2$, e vice-versa. Com negativos isso falha: $-5 < 2$, mas $(-5)^2 = 25 > 4 = 2^2$. Distâncias e raios nunca são negativos, então o truque é seguro.
</details>

> 📎 **Conexão.** O conjunto dos pontos que estão a uma distância fixa de um centro é a **circunferência**. No notebook 08 a condição $d_{JG}^2 = \text{raio}^2$ vai virar a **equação da circunferência**.

## 3. Ponto médio

O ponto médio $M$ de um segmento $AB$ é o ponto do segmento que fica à mesma distância de $A$ e de $B$.

**Na reta numérica**, o ponto médio entre dois números é a média deles: entre 2 e 8, o meio é $\frac{2 + 8}{2} = 5$.

**No plano**, vale a mesma ideia em cada eixo. Em 📎 [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) você já calculava o ponto médio assim, como média das coordenadas; o que faltava era o **porquê**. Projete $A$, $M$ e $B$ no eixo $x$. As projeções são retas paralelas (verticais) cortando duas transversais: o segmento $AB$ e o eixo $x$. Pelo **Teorema de Tales** (📎 [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb)), se $M$ divide $AB$ em duas partes iguais, a projeção de $M$ divide a projeção de $AB$ em duas partes iguais. Então a abscissa de $M$ é a média das abscissas. O mesmo raciocínio vale no eixo $y$.

> 📌 **Ponto médio.** Para $A(x_A, y_A)$ e $B(x_B, y_B)$:
> $$M = \left( \frac{x_A + x_B}{2},\ \frac{y_A + y_B}{2} \right)$$

Ou seja: **o ponto médio é a média dos pontos**, coordenada a coordenada.

🕹️ No widget abaixo, as projeções $A'$, $M'$ e $B'$ no eixo $x$ mostram Tales em ação: $M'$ fica sempre no meio de $A'$ e $B'$.

In [12]:
def escrever_ponto(P: Ponto) -> str:
    """Escreve um ponto com no máximo 4 casas decimais, como (2, 0.3333)."""
    # Somar 0.0 transforma um eventual "-0.0" em "0.0".
    return "(" + ", ".join(f"{round(coordenada, 4) + 0.0:g}" for coordenada in P) + ")"


def mostrar_ponto_medio(xA: int, yA: int, xB: int, yB: int) -> None:
    """Desenha o segmento AB, o ponto médio M e as projeções de A, M e B no eixo x."""
    A = (xA, yA)
    B = (xB, yB)

    # A conta do ponto médio: a média das coordenadas
    xM = (xA + xB) / 2
    yM = (yA + yB) / 2
    M = (xM, yM)

    fig = uga.criar_plano_cartesiano(titulo="O ponto médio M e as projeções no eixo x")
    # As projeções no eixo x: linhas verticais até o eixo (as paralelas de Tales)
    for P, nome in [(A, "A′"), (M, "M′"), (B, "B′")]:
        fig = uga.desenhar_segmento(
            fig, P, (P[0], 0), cor=uga.CORES["projecao"], espessura=1.5, tracejado=True
        )
        fig = uga.desenhar_ponto(
            fig, (P[0], 0), rotulo=nome, cor=uga.CORES["projecao"], tamanho=6,
            posicao_rotulo="bottom center",
        )
    fig = uga.desenhar_segmento(fig, A, B)
    fig = uga.desenhar_pontos(fig, [A, B], rotulos=["A", "B"], mostrar_coordenadas=True)
    fig = uga.desenhar_ponto(
        fig, M, rotulo="M", cor=uga.CORES["destaque"], mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    print(f"M = (({xA} + {xB})/2, ({yA} + {yB})/2) = {escrever_ponto(M)}")
    print(f"d(A, M) = {distancia(A, M):.4f}    d(M, B) = {distancia(M, B):.4f}")


interact(
    mostrar_ponto_medio,
    xA=controle("x_A", -3), yA=controle("y_A", 4),
    xB=controle("x_B", 5), yB=controle("y_B", 0),
);

interactive(children=(IntSlider(value=-3, continuous_update=False, description='x_A', max=5, min=-5, style=Sli…

In [13]:
def ponto_medio(A: Ponto, B: Ponto) -> Ponto:
    """Ponto médio do segmento AB."""
    xA, yA = A
    xB, yB = B
    return ((xA + xB) / 2, (yA + yB) / 2)


print(ponto_medio((-2, 5), (6, -1)))   # (2.0, 2.0)

(2.0, 2.0)


### O problema ao contrário: achar a outra ponta

Às vezes você conhece uma ponta $A$ e o ponto médio $M$, e quer a outra ponta $B$. Da fórmula, $x_M = \frac{x_A + x_B}{2}$, logo $x_B = 2x_M - x_A$ (e o mesmo para $y$):

$$B = (2x_M - x_A,\ 2y_M - y_A)$$

Dizemos que $B$ é o **simétrico de $A$ em relação a $M$**: $M$ funciona como um espelho em forma de ponto.

**Exemplo.** Se $A(1, 4)$ e $M(3, 1)$, então $B = (2 \cdot 3 - 1,\ 2 \cdot 1 - 4) = (5, -2)$.

### 3.1 Simetrias especiais

Algumas simetrias aparecem tanto que vale a pena guardar:

| Simétrico de $P(x, y)$ em relação a... | Resultado | Por quê |
|---|---|---|
| eixo $x$ | $(x, -y)$ | o eixo $x$ fica no meio do caminho entre os dois pontos |
| eixo $y$ | $(-x, y)$ | idem, com o eixo $y$ |
| origem $O$ | $(-x, -y)$ | $O$ é o ponto médio de $P$ e do simétrico |
| bissetriz dos quadrantes ímpares ($y = x$) | $(y, x)$ | as coordenadas trocam de lugar |
| bissetriz dos quadrantes pares ($y = -x$) | $(-y, -x)$ | trocam de lugar e de sinal |

A **bissetriz dos quadrantes ímpares** (1º e 3º) é formada pelos pontos com $x = y$, como $(2, 2)$ e $(-3, -3)$. A **bissetriz dos quadrantes pares** (2º e 4º) é formada pelos pontos com $y = -x$, como $(-2, 2)$ e $(3, -3)$.

> 📎 **Conexão.** A troca $(x, y) \to (y, x)$ você já viu no módulo 01: é o que acontece com os pares de uma **relação inversa** (📎 [`0104a_relacoes.ipynb`](../01_conjuntos_e_funcoes/0104a_relacoes.ipynb)), cujo gráfico é o reflexo do original na reta $y = x$. Agora você sabe por que esse reflexo funciona: a reta $y = x$ fica exatamente no meio do caminho entre $(x, y)$ e $(y, x)$, porque o ponto médio entre eles tem as duas coordenadas iguais.

🕹️ Escolha o ponto $P$ e o "espelho" no widget abaixo.

In [ ]:
ESPELHOS = [
    ("eixo x", "eixo_x"),
    ("eixo y", "eixo_y"),
    ("origem", "origem"),
    ("bissetriz dos quadrantes ímpares (y = x)", "impares"),
    ("bissetriz dos quadrantes pares (y = −x)", "pares"),
]


def simetrico(P: Ponto, espelho: str) -> Ponto:
    """O simétrico de P em relação ao espelho escolhido (a tabela da seção 3.1)."""
    x, y = P
    if espelho == "eixo_x":
        return (x, -y)
    if espelho == "eixo_y":
        return (-x, y)
    if espelho == "origem":
        return (-x, -y)
    if espelho == "impares":
        return (y, x)
    return (-y, -x)   # bissetriz dos quadrantes pares


def mostrar_simetria(x: int, y: int, espelho: str) -> None:
    """Desenha P, o simétrico P′, o espelho e o ponto médio de PP′."""
    P = (x, y)
    P_linha = simetrico(P, espelho)
    M = ponto_medio(P, P_linha)

    fig = uga.criar_plano_cartesiano(titulo="P e o seu simétrico P′")
    # O espelho, em destaque
    cor_espelho = uga.CORES["reta_secundaria"]
    if espelho == "eixo_x":
        fig = uga.desenhar_segmento(fig, (-6, 0), (6, 0), cor=cor_espelho, espessura=4)
    elif espelho == "eixo_y":
        fig = uga.desenhar_segmento(fig, (0, -6), (0, 6), cor=cor_espelho, espessura=4)
    elif espelho == "origem":
        fig = uga.desenhar_ponto(fig, (0, 0), rotulo="O", cor=cor_espelho, tamanho=14)
    elif espelho == "impares":
        fig = uga.desenhar_segmento(fig, (-6, -6), (6, 6), cor=cor_espelho, espessura=4)
        fig = uga.adicionar_rotulo(fig, (4.3, 5.3), "y = x", cor=cor_espelho)
    else:
        fig = uga.desenhar_segmento(fig, (-6, 6), (6, -6), cor=cor_espelho, espessura=4)
        fig = uga.adicionar_rotulo(fig, (-4.3, 5.3), "y = −x", cor=cor_espelho)

    fig = uga.desenhar_segmento(fig, P, P_linha, cor=uga.CORES["apoio"], tracejado=True)
    fig = uga.desenhar_ponto(fig, M, cor=uga.CORES["apoio"], tamanho=7)
    fig = uga.desenhar_ponto(fig, P, rotulo="P", mostrar_coordenadas=True)
    fig = uga.desenhar_ponto(
        fig, P_linha, rotulo="P′", cor=uga.CORES["destaque"], mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    print(
        f"P = {escrever_ponto(P)} → P′ = {escrever_ponto(P_linha)}; "
        f"ponto médio de PP′ = {escrever_ponto(M)}, que está sobre o espelho."
    )
    if P == P_linha:
        print("(P está sobre o espelho: ele é o seu próprio simétrico.)")


interact(
    mostrar_simetria,
    x=controle("x de P", 4), y=controle("y de P", 2),
    espelho=widgets.Dropdown(
        options=ESPELHOS, value="impares", description="espelho",
        style={"description_width": "initial"},
    ),
);

interactive(children=(IntSlider(value=4, continuous_update=False, description='x de P', max=5, min=-5, style=S…

**Pare e pense**

1. Qual ponto é simétrico de si mesmo em relação à origem?
2. Qual é o simétrico de $(3, 3)$ em relação à bissetriz dos quadrantes ímpares? Por que isso faz sentido?

<details>
<summary>Ver respostas</summary>

1. Só a própria origem $(0, 0)$.
2. É o próprio $(3, 3)$: o ponto já está sobre o espelho.
</details>

## 4. Andando sobre um segmento

O ponto médio é o ponto que está na **metade** do caminho de $A$ até $B$. E o ponto que está a **um terço** do caminho? Ou a 70% do caminho?

Pense numa viagem de $A$ até $B$. Para ir de $A$ até $B$, você anda $\Delta x$ na horizontal e $\Delta y$ na vertical. Se você percorreu apenas a fração $t$ da viagem (por exemplo, $t = 0{,}3$ para 30%), andou só $t \cdot \Delta x$ na horizontal e $t \cdot \Delta y$ na vertical. Pela semelhança de triângulos (📎 [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb)), o triângulo retângulo dessa parte da viagem é uma cópia reduzida do triângulo completo.

> 📌 **Ponto que está na fração $t$ do caminho de $A$ até $B$** ($0 \le t \le 1$):
> $$P = \big(x_A + t\,(x_B - x_A),\ \ y_A + t\,(y_B - y_A)\big)$$

Confira os casos especiais:
- $t = 0$ → $P = A$ (ainda não saiu);
- $t = 1$ → $P = B$ (chegou);
- $t = \frac{1}{2}$ → $P = \left(x_A + \frac{x_B - x_A}{2},\ \dots\right) = \left(\frac{x_A + x_B}{2},\ \dots\right)$, o ponto médio!

🕹️ No widget abaixo, o triângulo pequeno (a parte já percorrida) é uma cópia reduzida do triângulo grande (a viagem inteira).

In [ ]:
def mostrar_caminho(xA: int, yA: int, xB: int, yB: int, t: float) -> None:
    """Desenha o ponto P que está na fração t do caminho de A até B."""
    A = (xA, yA)
    B = (xB, yB)

    # A conta: andamos a fração t de Δx e a fração t de Δy
    delta_x = xB - xA
    delta_y = yB - yA
    P = (xA + t * delta_x, yA + t * delta_y)

    fig = uga.criar_plano_cartesiano(titulo=f"P está na fração t = {t:.2f} do caminho")
    fig = uga.desenhar_triangulo_das_diferencas(fig, A, B)                 # a viagem inteira
    fig = uga.desenhar_triangulo_das_diferencas(fig, A, P, rotulos=False)  # a parte percorrida
    fig = uga.desenhar_segmento(fig, A, B, cor=uga.CORES["apoio"], espessura=2)
    fig = uga.desenhar_segmento(fig, A, P, cor=uga.CORES["reta"], espessura=6)
    fig = uga.desenhar_pontos(fig, [A, B], rotulos=["A", "B"])
    fig = uga.desenhar_ponto(
        fig, P, rotulo="P", cor=uga.CORES["destaque"], mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    AP = distancia(A, P)
    AB = distancia(A, B)
    if AB == 0:
        print("A e B coincidem: não há caminho para andar.")
        return
    print(
        f"t = {t:.2f} → P = {escrever_ponto(P)}; "
        f"AP = {AP:.2f} e AB = {AB:.2f}; AP/AB = {AP / AB:.2f}"
    )


interact(
    mostrar_caminho,
    xA=controle("x_A", -4), yA=controle("y_A", -2),
    xB=controle("x_B", 4), yB=controle("y_B", 3),
    t=widgets.FloatSlider(
        min=0, max=1, step=0.05, value=0.3, description="t",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=-4, continuous_update=False, description='x_A', max=5, min=-5, style=Sli…

### Dividir na razão $k$

Muitos livros falam em "o ponto $P$ que divide $AB$ na razão $\frac{AP}{PB} = k$". É a mesma coisa, só que contando de outro jeito. Se $AP$ é $k$ vezes $PB$, o segmento inteiro tem $k + 1$ "pedaços" iguais, e $P$ está depois de $k$ deles:

$$t = \frac{k}{k + 1}$$

Por exemplo, a razão $\frac{AP}{PB} = 2$ corresponde a $t = \frac{2}{3}$: o ponto está a dois terços do caminho.

> 📎 **Conexão.** Foi assim que você dividiu segmentos em [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb) (seção 1): com a razão $m : n$, o ponto era $P = A + \frac{m}{m + n}(B - A)$. É a mesma fórmula, com $t = \frac{m}{m + n}$. A novidade aqui é pensar em $t$ como **um único número** que diz quanto do caminho já foi percorrido.

### Interpolação linear: como os personagens andam

> 🎯 **Aplicação prática.** Quando um personagem de jogo ou de animação vai de $A$ até $B$, o programa calcula a posição dele em cada quadro com exatamente essa fórmula, aumentando $t$ um pouquinho de cada vez. Os programadores chamam isso de **interpolação linear** (em inglês, *lerp*).

> 📎 **Conexão.** Se você deixar $t$ ser **qualquer** número real (negativo ou maior que 1), o ponto $P$ sai do segmento e percorre a **reta** inteira que passa por $A$ e $B$. Essa ideia vai virar a **equação paramétrica da reta** no notebook 04.

In [16]:
def ponto_do_segmento(A: Ponto, B: Ponto, t: float) -> Ponto:
    """Ponto que está na fração t do caminho de A até B."""
    xA, yA = A
    xB, yB = B
    return (xA + t * (xB - xA), yA + t * (yB - yA))


# Dividindo AB em 3 partes iguais
A, B = (-1, 2), (8, -4)
print(ponto_do_segmento(A, B, 1/3))   # (2.0, 0.0)
print(ponto_do_segmento(A, B, 2/3))   # (5.0, -2.0)

(2.0, 0.0)
(5.0, -2.0)


🕹️ Aperte o botão **▶ Andar** na figura abaixo para ver um personagem indo de $A$ até $B$, quadro a quadro.

In [17]:
A_animacao, B_animacao = (-4, -2), (4, 3)
valores_de_t = [k / 20 for k in range(21)]   # t = 0; 0,05; 0,1; ...; 1

# O cenário: o segmento, as pontas e o ponto P (que começa em A)
fig = uga.criar_plano_cartesiano(titulo="Interpolação linear: P andando de A até B")
fig = uga.desenhar_segmento(fig, A_animacao, B_animacao, cor=uga.CORES["apoio"])
fig = uga.desenhar_pontos(
    fig, [A_animacao, B_animacao], rotulos=["A", "B"], mostrar_coordenadas=True
)
fig = uga.desenhar_ponto(
    fig, A_animacao, rotulo="P", cor=uga.CORES["destaque"], mostrar_coordenadas=True
)
indice_de_P = len(fig.data) - 1   # P é o último desenho da figura

# Os quadros da animação
quadros = []
for t in valores_de_t:
    # Cada quadro é a fórmula da interpolação linear com um t um pouco maior.
    P = ponto_do_segmento(A_animacao, B_animacao, t)
    # Desenhamos P numa figura de rascunho só para aproveitar o visual do uga.
    rascunho = uga.desenhar_ponto(
        uga.criar_plano_cartesiano(), P, rotulo="P", cor=uga.CORES["destaque"],
        mostrar_coordenadas=True,
    )
    quadros.append({"data": [rascunho.data[0]], "traces": [indice_de_P], "name": f"{t:.2f}"})
fig.frames = quadros

# O botão que toca a animação, um quadro a cada 150 milissegundos
fig.update_layout(updatemenus=[{
    "type": "buttons", "showactive": False,
    "x": 0, "y": 1.02, "xanchor": "left", "yanchor": "bottom",
    "bgcolor": uga.NORD["nord1"], "bordercolor": uga.NORD["nord3"],
    "font": {"color": uga.CORES["texto"]},
    "buttons": [{
        "label": "▶ Andar", "method": "animate",
        "args": [None, {"frame": {"duration": 150, "redraw": False},
                        "transition": {"duration": 0}, "fromcurrent": False}],
    }],
}])
uga.mostrar(fig)

# A mesma viagem, em números
print(f"{'t':>5} | {'x_P':>6} | {'y_P':>6}")
for t in [0, 0.25, 0.5, 0.75, 1]:
    x_P, y_P = ponto_do_segmento(A_animacao, B_animacao, t)
    print(f"{t:>5} | {x_P:>6} | {y_P:>6}")

    t |    x_P |    y_P
    0 |     -4 |     -2
 0.25 |   -2.0 |  -0.75
  0.5 |    0.0 |    0.5
 0.75 |    2.0 |   1.75
    1 |      4 |      3


**Pare e pense**

1. Que valor de $t$ dá o ponto que divide $AB$ na razão $\frac{AP}{PB} = \frac{1}{3}$?
2. O ponto com $t = 0{,}9$ está mais perto de $A$ ou de $B$?

<details>
<summary>Ver respostas</summary>

1. $t = \frac{1/3}{1/3 + 1} = \frac{1}{4}$: o ponto está a um quarto do caminho.
2. De $B$: ele já percorreu 90% do caminho.
</details>

## 5. Baricentro

No módulo 04 (📎 [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb)) você estudou o **baricentro** $G$: o ponto onde as três **medianas** de um triângulo se encontram. E provou uma propriedade importante: **o baricentro divide cada mediana na razão 2 : 1**, contando a partir do vértice (a demonstração sintética, com bases médias, veio em 📎 [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb)).

Lá mesmo, com coordenadas, você chegou à fórmula de $G$. Não vamos provar nada disso de novo: vamos **usar** a propriedade e refazer a conta em três linhas, agora com as duas ferramentas que você acabou de programar (o ponto médio e a fração do caminho).

Tome a mediana que sai de $A$ e vai até $M_a$, o ponto médio de $BC$:

1. Pelo ponto médio: $M_a = \left(\frac{x_B + x_C}{2},\ \frac{y_B + y_C}{2}\right)$.
2. Pela razão 2 : 1, o baricentro está a $\frac{2}{3}$ do caminho de $A$ até $M_a$. Então, para a abscissa:
$$x_G = x_A + \tfrac{2}{3}\left(\tfrac{x_B + x_C}{2} - x_A\right) = x_A + \tfrac{x_B + x_C}{3} - \tfrac{2x_A}{3} = \tfrac{x_A + x_B + x_C}{3}$$
3. A conta da ordenada é igual.

> 📌 **Baricentro.** Para o triângulo de vértices $A$, $B$ e $C$:
> $$G = \left( \frac{x_A + x_B + x_C}{3},\ \frac{y_A + y_B + y_C}{3} \right)$$

O ponto médio era a média de **dois** pontos; o baricentro é a média dos **três** vértices.

> 📎 **Conexão (outra caixa que se abre).** No notebook 17 do módulo 06 (📎 [`0617a_superficies_e_solidos_de_revolucao.ipynb`](../06_geometria_espacial/0617a_superficies_e_solidos_de_revolucao.ipynb)), a função `u3d.medir_area_e_centroide_2d` calculava o centroide de regiões para você usar no Teorema de Pappus. Lembra daquela função que você usou no módulo 06? Agora você sabe como ela funciona para um triângulo: o centroide que ela devolvia é exatamente esta média dos vértices. (Para polígonos quaisquer, a conta interna dela usa a fórmula do cadarço, que você vai abrir no notebook 02.)

> 🎯 **Aplicação prática.** Na física, o baricentro é o **centro de massa** de uma placa triangular fina e homogênea: é o ponto onde você consegue equilibrar o triângulo na ponta de um lápis.

🕹️ Mova os vértices e confira: as três medianas sempre passam pelo mesmo ponto.

In [18]:
def baricentro(A: Ponto, B: Ponto, C: Ponto) -> Ponto:
    """Baricentro do triângulo ABC: a média dos três vértices."""
    xA, yA = A
    xB, yB = B
    xC, yC = C
    return ((xA + xB + xC) / 3, (yA + yB + yC) / 3)


def mesmo_lugar(P: Ponto, Q: Ponto) -> bool:
    """True se P e Q são o mesmo ponto (a menos de erros de arredondamento)."""
    return math.isclose(P[0], Q[0], abs_tol=1e-9) and math.isclose(P[1], Q[1], abs_tol=1e-9)


def mostrar_baricentro(xA: int, yA: int, xB: int, yB: int, xC: int, yC: int) -> None:
    """Desenha o triângulo, as três medianas e o baricentro G, e confere as contas."""
    A, B, C = (xA, yA), (xB, yB), (xC, yC)

    # Três pontos alinhados não formam triângulo: o maior "lado" seria igual à soma
    # dos outros dois (a desigualdade triangular vira uma igualdade).
    lados = sorted([distancia(A, B), distancia(B, C), distancia(C, A)])
    if math.isclose(lados[0] + lados[1], lados[2]):
        fig = uga.criar_plano_cartesiano(titulo="Três pontos alinhados")
        fig = uga.desenhar_pontos(fig, [A, B, C], rotulos=["A", "B", "C"])
        uga.mostrar(fig)
        print("Os três pontos estão alinhados: não formam um triângulo. "
              "(No notebook 02 você vai aprender a detectar isso com uma conta.)")
        return

    # G pela fórmula: a média dos vértices
    G = baricentro(A, B, C)

    # G pelas medianas: em cada uma, o ponto a 2/3 do caminho do vértice até o
    # ponto médio do lado oposto
    medianas = [("a", A, ponto_medio(B, C)), ("b", B, ponto_medio(C, A)),
                ("c", C, ponto_medio(A, B))]
    pontos_pelas_medianas = [ponto_do_segmento(V, M, 2 / 3) for _, V, M in medianas]

    fig = uga.criar_plano_cartesiano(titulo="As três medianas e o baricentro G")
    fig = uga.desenhar_poligono(fig, [A, B, C], rotulos=["A", "B", "C"])
    for nome, V, M in medianas:
        fig = uga.desenhar_segmento(fig, V, M, espessura=2, tracejado=True)
        fig = uga.desenhar_ponto(fig, M, rotulo=f"M{nome}", cor=uga.CORES["apoio"], tamanho=7)
    fig = uga.desenhar_ponto(
        fig, G, rotulo="G", cor=uga.CORES["destaque"], tamanho=11, mostrar_coordenadas=True
    )
    uga.mostrar(fig)

    for (nome, _, _), P in zip(medianas, pontos_pelas_medianas):
        print(f"Pela mediana que sai de {nome.upper()}: {escrever_ponto(P)}")
    print(f"Pela fórmula (média dos vértices):  G = {escrever_ponto(G)}")
    if all(mesmo_lugar(P, G) for P in pontos_pelas_medianas):
        print("✅ As três medianas passam pelo mesmo ponto G.")


interact(
    mostrar_baricentro,
    xA=controle("x_A", -4), yA=controle("y_A", -3),
    xB=controle("x_B", 5), yB=controle("y_B", -1),
    xC=controle("x_C", 0), yC=controle("y_C", 4),
);

interactive(children=(IntSlider(value=-4, continuous_update=False, description='x_A', max=5, min=-5, style=Sli…

**Pare e pense**

1. Um triângulo tem vértices $A(0, 0)$, $B(6, 0)$ e $C(0, 9)$. Sem desenhar, onde está o baricentro?
2. Se você somar 1 à abscissa dos três vértices (deslocando o triângulo para a direita), o que acontece com $G$?

<details>
<summary>Ver respostas</summary>

1. $G = \left(\frac{0 + 6 + 0}{3},\ \frac{0 + 0 + 9}{3}\right) = (2, 3)$.
2. $G$ também anda 1 para a direita: a soma das abscissas aumenta 3, e a média aumenta 1. O baricentro "acompanha" o triângulo.
</details>

## 6. Provando teoremas com coordenadas

Até aqui, você usou coordenadas para **calcular**. Mas elas também servem para **provar**. Vamos comparar os dois métodos num teorema que você já conhece do módulo 04:

> **Teorema.** Num triângulo retângulo, o ponto médio da hipotenusa está à mesma distância dos três vértices.

### O jeito sintético (módulo 04), resumido

📎 [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (seção 3) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (seção 5). Você completava o triângulo retângulo $ABC$ até formar um retângulo. As diagonais de um retângulo são iguais e se cortam ao meio, então o ponto médio da hipotenusa é o centro do retângulo e está à mesma distância dos quatro cantos. Uma ideia esperta: **enxergar o retângulo escondido**.

### O jeito analítico

1. **Escolha bem os eixos.** Coloque o ângulo reto na origem e os catetos sobre os eixos: $A(0, 0)$, $B(b, 0)$ e $C(0, c)$, com $b > 0$ e $c > 0$. Como $b$ e $c$ são letras, isso representa **qualquer** triângulo retângulo.
2. **Calcule o ponto médio da hipotenusa $BC$:** $M = \left(\frac{b}{2}, \frac{c}{2}\right)$.
3. **Calcule as três distâncias:**
$$d_{MA} = \sqrt{\left(\tfrac{b}{2}\right)^2 + \left(\tfrac{c}{2}\right)^2} = \tfrac{1}{2}\sqrt{b^2 + c^2}$$
$$d_{MB} = \sqrt{\left(b - \tfrac{b}{2}\right)^2 + \left(0 - \tfrac{c}{2}\right)^2} = \tfrac{1}{2}\sqrt{b^2 + c^2}$$
$$d_{MC} = \sqrt{\left(0 - \tfrac{b}{2}\right)^2 + \left(c - \tfrac{c}{2}\right)^2} = \tfrac{1}{2}\sqrt{b^2 + c^2}$$

As três são iguais. ∎ Nenhuma ideia esperta foi necessária: só **escolher os eixos e fazer as contas**.

> 💡 **Dica.** Escolher bem a posição dos eixos é a "ideia esperta" do método analítico. Coloque um vértice na origem e um lado sobre um eixo sempre que puder: as contas ficam muito mais curtas.

### Testar não é provar

O computador consegue testar o teorema em milhares de triângulos em um piscar de olhos. Isso **convence**, mas **não prova**: talvez exista um triângulo estranho que nunca sorteamos. A prova acima, com as letras $b$ e $c$, vale para **todos** os triângulos retângulos de uma vez.

Mesmo assim, testar é muito útil: se um teste falhar, você descobre na hora que a afirmação é falsa (ou que errou alguma conta).

In [19]:
import random

random.seed(2026)   # semente fixa: a cada execução, os mesmos números são sorteados

# 1000 triângulos retângulos, com o ângulo reto na origem
confirmados = 0
for _ in range(1000):
    b = random.uniform(0.1, 100)
    c = random.uniform(0.1, 100)
    A, B, C = (0, 0), (b, 0), (0, c)
    M = ponto_medio(B, C)   # o ponto médio da hipotenusa
    d_MA, d_MB, d_MC = distancia(M, A), distancia(M, B), distancia(M, C)
    if math.isclose(d_MA, d_MB) and math.isclose(d_MA, d_MC):
        confirmados += 1
print(f"Triângulos testados: 1000 · Teorema confirmado em: {confirmados}")

# 1000 triângulos quaisquer: M continua sendo o ponto médio de BC
confirmados_quaisquer = 0
for _ in range(1000):
    A, B, C = [(random.uniform(-10, 10), random.uniform(-10, 10)) for _ in range(3)]
    M = ponto_medio(B, C)
    d_MA, d_MB, d_MC = distancia(M, A), distancia(M, B), distancia(M, C)
    if math.isclose(d_MA, d_MB) and math.isclose(d_MA, d_MC):
        confirmados_quaisquer += 1
print(f"Em triângulos quaisquer, confirmado em: {confirmados_quaisquer}")

Triângulos testados: 1000 · Teorema confirmado em: 1000
Em triângulos quaisquer, confirmado em: 0


Nos triângulos quaisquer, a afirmação falha praticamente sempre: a hipótese "**retângulo**" é essencial. Nos triângulos retângulos, ela vale em todos os testes, como a prova garante.

#### Para curiosos: o computador também sabe fazer contas com letras

A biblioteca `sympy` faz **álgebra com símbolos**, como você faz no papel. Com ela, o computador não testa exemplos: ele refaz a prova com as letras $b$ e $c$. Você não precisa saber usar o `sympy` para continuar o curso; esta célula é só uma amostra.

In [20]:
import sympy

b, c = sympy.symbols("b c", positive=True)   # duas letras que representam números positivos
A, B, C = (0, 0), (b, 0), (0, c)

# A nossa função ponto_medio funciona até com letras!
M = ponto_medio(B, C)


def distancia_simbolica(P: tuple, Q: tuple) -> sympy.Expr:
    """A fórmula da distância, com a raiz do sympy (que sabe lidar com letras)."""
    return sympy.sqrt((Q[0] - P[0])**2 + (Q[1] - P[1])**2)


d_MA = distancia_simbolica(M, A)
d_MB = distancia_simbolica(M, B)
d_MC = distancia_simbolica(M, C)

print("M =", M)
print("d(M, A) =", d_MA)
print("d(M, B) =", d_MB)
print("d(M, C) =", d_MC)
print("d(M, A) − d(M, B) simplificado:", sympy.simplify(d_MA - d_MB))
print("d(M, A) − d(M, C) simplificado:", sympy.simplify(d_MA - d_MC))

M = (b/2, c/2)
d(M, A) = sqrt(b**2/4 + c**2/4)
d(M, B) = sqrt(b**2/4 + c**2/4)
d(M, C) = sqrt(b**2/4 + c**2/4)
d(M, A) − d(M, B) simplificado: 0
d(M, A) − d(M, C) simplificado: 0


**Pare e pense**

Na prova analítica, por que podemos colocar o triângulo com o ângulo reto na origem? Isso não deixa a prova "menos geral"?

<details>
<summary>Ver resposta</summary>

Não deixa. Qualquer triângulo retângulo pode ser movido e girado até ficar nessa posição, e mover ou girar uma figura não muda as distâncias entre seus pontos. Escolher os eixos é só escolher de onde olhar para a figura.
</details>

| | Sintético | Analítico |
|---|---|---|
| Ideia principal | enxergar o retângulo escondido | escolher os eixos |
| Trabalho | pouco cálculo, muita criatividade | pouca criatividade, mais cálculo |
| O computador ajuda? | quase nada | muito (testes e até a própria prova) |

## Resumo

**A grande ideia:** a geometria analítica descreve figuras por números e resolve problemas com contas. Ela não substitui a geometria sintética dos módulos 04 e 06: as duas se completam.

**Fórmulas** (para $A(x_A, y_A)$, $B(x_B, y_B)$, $C(x_C, y_C)$)

| O quê | Fórmula |
|---|---|
| Distância | $d_{AB} = \sqrt{(x_B - x_A)^2 + (y_B - y_A)^2}$ |
| Ponto médio | $M = \left(\frac{x_A + x_B}{2},\ \frac{y_A + y_B}{2}\right)$ |
| Outra ponta, dados $A$ e $M$ | $B = (2x_M - x_A,\ 2y_M - y_A)$ |
| Ponto na fração $t$ do caminho | $P = (x_A + t\,\Delta x,\ y_A + t\,\Delta y)$ |
| Razão $\frac{AP}{PB} = k$ | $t = \frac{k}{k + 1}$ |
| Baricentro | $G = \left(\frac{x_A + x_B + x_C}{3},\ \frac{y_A + y_B + y_C}{3}\right)$ |

**Simetrias de $P(x, y)$:** eixo $x$ → $(x, -y)$ · eixo $y$ → $(-x, y)$ · origem → $(-x, -y)$ · $y = x$ → $(y, x)$ · $y = -x$ → $(-y, -x)$.

**Classificar um triângulo pelos vértices:** calcule os **quadrados** dos lados e use a recíproca de Pitágoras com o maior lado.

**Truques:** compare quadrados de distâncias para evitar raízes; nas provas, coloque um vértice na origem e um lado sobre um eixo.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Você pode usar as funções que programou neste notebook (`distancia`, `ponto_medio`, `ponto_do_segmento`, `baricentro`, `classificar_triangulo`). Escreva os pontos como tuplas, por exemplo `(2, -1)`.

Rode também a célula abaixo: ela prepara a ferramenta que as verificações usam para comparar pontos.

In [18]:
def mesmo_ponto(resposta, certo: Ponto) -> bool:
    """Confere se a resposta é o ponto certo, aceitando pequenos erros de arredondamento.

    Usada nas células de verificação: compara coordenada a coordenada com math.isclose.
    """
    if resposta is ...:
        return False
    try:
        return len(resposta) == len(certo) and all(
            math.isclose(r, c, abs_tol=1e-9) for r, c in zip(resposta, certo)
        )
    except TypeError:   # a resposta não é uma lista de números (por exemplo, um número só)
        return False

### Básico

In [ ]:
# Exercício Básico 1: distâncias. Calcule:
# (a) a distância entre (1, 2) e (4, 6);
# (b) a distância entre (-3, -1) e (2, 11);
# (c) a distância entre (0, 0) e (-6, 8).
distancia_a_ex1 = ...
distancia_b_ex1 = ...
distancia_c_ex1 = ...

print(distancia_a_ex1, distancia_b_ex1, distancia_c_ex1)

In [ ]:
# Verificação
respostas_ex1 = [("a", distancia_a_ex1, 5), ("b", distancia_b_ex1, 13), ("c", distancia_c_ex1, 10)]
for item, resposta, certa in respostas_ex1:
    assert resposta is not ... and math.isclose(resposta, certa), (
        f"Tente de novo, revise: no item ({item}), cuidado com os sinais ao subtrair "
        "números negativos (seção 1 - Distância entre dois pontos)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: ponto médio e outra ponta.
# (a) Qual é o ponto médio de A(-2, 5) e B(6, -1)?
# (b) M(3, 1) é o ponto médio de AB, com A(1, 4). Encontre B.
ponto_medio_ex2 = ...
ponta_b_ex2 = ...

print(ponto_medio_ex2, ponta_b_ex2)

In [ ]:
# Verificação
assert mesmo_ponto(ponto_medio_ex2, (2, 2)), (
    "Tente de novo, revise: no item (a), o ponto médio é a média das coordenadas "
    "(seção 3 - Ponto médio)."
)
assert mesmo_ponto(ponta_b_ex2, (5, -2)), (
    "Tente de novo, revise: no item (b), use B = (2·xM − xA, 2·yM − yA) "
    "(seção 3 - Ponto médio)."
)
print("Certo!")

In [ ]:
# Exercício Básico 3: simetrias. Para P(3, -5), dê o simétrico em relação:
# (a) ao eixo x;
# (b) ao eixo y;
# (c) à origem;
# (d) à bissetriz dos quadrantes ímpares.
simetrico_a_ex3 = ...
simetrico_b_ex3 = ...
simetrico_c_ex3 = ...
simetrico_d_ex3 = ...

print(simetrico_a_ex3, simetrico_b_ex3, simetrico_c_ex3, simetrico_d_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", simetrico_a_ex3, (3, 5)),
    ("b", simetrico_b_ex3, (-3, -5)),
    ("c", simetrico_c_ex3, (-3, 5)),
    ("d", simetrico_d_ex3, (-5, 3)),
]
for item, resposta, certa in respostas_ex3:
    assert mesmo_ponto(resposta, certa), (
        f"Tente de novo, revise: o item ({item}) não está certo; confira a tabela "
        "(seção 3.1 - Simetrias especiais)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: classifique sem desenhar. Para cada triângulo, diga a
# classificação quanto aos lados ("equilátero", "isósceles" ou "escaleno") e quanto
# aos ângulos ("acutângulo", "retângulo" ou "obtusângulo"), em letras minúsculas e
# com acento.
# (a) A(0, 0), B(4, 0), C(1, 3);
# (b) A(1, 1), B(4, 5), C(5, -2).
lados_a_ex4 = ...
angulos_a_ex4 = ...
lados_b_ex4 = ...
angulos_b_ex4 = ...

print(lados_a_ex4, angulos_a_ex4, lados_b_ex4, angulos_b_ex4)

In [ ]:
# Verificação
respostas_ex4 = [
    ("a", lados_a_ex4, "escaleno"), ("a", angulos_a_ex4, "acutângulo"),
    ("b", lados_b_ex4, "isósceles"), ("b", angulos_b_ex4, "retângulo"),
]
for item, resposta, certa in respostas_ex4:
    assert isinstance(resposta, str) and resposta.strip().lower() == certa, (
        f"Tente de novo, revise: no item ({item}), trabalhe com os quadrados dos lados "
        "e compare o maior com a soma dos outros dois "
        "(seção 2.1 - Classificar um triângulo sem olhar para ele)."
    )
print("Certo!")

In [ ]:
# Exercício Intermediário 5: equidistante. Encontre o ponto P do eixo x que está à
# mesma distância de A(1, 3) e de B(5, 1).
# Sugestão: escreva P(x, 0), iguale os quadrados das distâncias e resolva a equação;
# repare que o x² some.
ponto_p_ex5 = ...

print(ponto_p_ex5)

In [ ]:
# Verificação
assert mesmo_ponto(ponto_p_ex5, (2, 0)), (
    "Tente de novo, revise: iguale d(P, A)² e d(P, B)² e simplifique: a equação fica "
    "do 1º grau (seção 1 - Distância entre dois pontos)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: dividindo em três. Encontre os dois pontos que dividem o
# segmento de A(-1, 2) a B(8, -4) em três partes iguais:
# (a) o mais perto de A;
# (b) o mais perto de B.
ponto_perto_de_a_ex6 = ...
ponto_perto_de_b_ex6 = ...

print(ponto_perto_de_a_ex6, ponto_perto_de_b_ex6)

In [ ]:
# Verificação
assert mesmo_ponto(ponto_perto_de_a_ex6, (2, 0)), (
    "Tente de novo, revise: no item (a), use t = 1/3 (seção 4 - Andando sobre um segmento)."
)
assert mesmo_ponto(ponto_perto_de_b_ex6, (5, -2)), (
    "Tente de novo, revise: no item (b), use t = 2/3 (seção 4 - Andando sobre um segmento)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 7: baricentro, nos dois sentidos.
# (a) Qual é o baricentro do triângulo A(2, 1), B(7, 3), C(3, 8)?
# (b) Um triângulo tem baricentro G(1, 2) e dois vértices A(0, 0) e B(5, 1).
#     Encontre o terceiro vértice C.
baricentro_ex7 = ...
vertice_c_ex7 = ...

print(baricentro_ex7, vertice_c_ex7)

In [ ]:
# Verificação
assert mesmo_ponto(baricentro_ex7, (4, 4)), (
    "Tente de novo, revise: no item (a), G é a média dos três vértices (seção 5 - Baricentro)."
)
assert mesmo_ponto(vertice_c_ex7, (-2, 5)), (
    "Tente de novo, revise: no item (b), 3·xG = xA + xB + xC (e o mesmo para y) "
    "(seção 5 - Baricentro)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 8: o quarto vértice. A(1, 1), B(5, 2) e C(6, 5) são três vértices
# consecutivos do paralelogramo ABCD. Encontre D.
# Lembre-se (📎 0409a_paralelogramos_e_casos_particulares.ipynb): as diagonais de um
# paralelogramo se cortam ao meio.
vertice_d_ex8 = ...

print(vertice_d_ex8)

In [ ]:
# Verificação
assert mesmo_ponto(vertice_d_ex8, (2, 4)), (
    "Tente de novo, revise: o ponto médio de AC é o mesmo que o de BD; calcule M de AC e "
    "depois a outra ponta D, a partir de B e M (seção 3 - Ponto médio)."
)
print("Certo!")

Depois de acertar, rode a célula abaixo para ver o paralelogramo com as duas diagonais e o ponto médio comum.

In [ ]:
# Verificação visual do Exercício Desafio 8 (rode depois de acertar).
assert vertice_d_ex8 is not ..., (
    "Tente de novo, revise: primeiro resolva o exercício acima (seção 3 - Ponto médio)."
)
A8, B8, C8, D8 = (1, 1), (5, 2), (6, 5), vertice_d_ex8
fig = uga.criar_plano_cartesiano(
    titulo="O paralelogramo ABCD e as suas diagonais", x_min=-1, x_max=8, y_min=-1, y_max=7
)
fig = uga.desenhar_poligono(fig, [A8, B8, C8, D8], rotulos=["A", "B", "C", "D"])
fig = uga.desenhar_segmento(fig, A8, C8, espessura=2, tracejado=True)
fig = uga.desenhar_segmento(fig, B8, D8, espessura=2, tracejado=True)
fig = uga.desenhar_ponto(
    fig, ponto_medio(A8, C8), rotulo="M", cor=uga.CORES["destaque"], mostrar_coordenadas=True
)
uga.mostrar(fig)
print(f"Ponto médio de AC: {escrever_ponto(ponto_medio(A8, C8))}")
print(f"Ponto médio de BD: {escrever_ponto(ponto_medio(B8, D8))}")

In [ ]:
# Exercício Desafio 9: a base média, pelo método analítico. No módulo 04
# (📎 0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) você viu que o
# segmento que liga os pontos médios de dois lados de um triângulo mede metade do
# terceiro lado.
# (a) Complete a função testar_base_media(A, B, C): ela deve devolver True se o
#     segmento que liga os pontos médios de AC e de BC mede metade de AB. Use
#     ponto_medio (seção 3), distancia (seção 1) e math.isclose.
#     Depois, teste a função em 1000 triângulos sorteados (vértices com coordenadas
#     entre -10 e 10) e guarde em todos_ok_ex9 o valor True se ela deu True em todos.
import random


def testar_base_media(A: Ponto, B: Ponto, C: Ponto) -> bool:
    ...  # complete aqui


todos_ok_ex9 = ...  # complete aqui: sorteie 1000 triângulos e teste cada um

print(todos_ok_ex9)

In [ ]:
# Verificação
import random

assert todos_ok_ex9 is True, (
    "Tente de novo, revise: guarde em todos_ok_ex9 o valor True se a função deu True nos "
    "1000 triângulos (seção 6 - Provando teoremas com coordenadas)."
)
triangulos_de_teste = [
    [(random.uniform(-10, 10), random.uniform(-10, 10)) for _ in range(3)] for _ in range(200)
]
assert all(testar_base_media(*T) for T in triangulos_de_teste), (
    "Tente de novo, revise: a função deve devolver True em todo triângulo; compare a "
    "distância entre os dois pontos médios com metade de AB usando math.isclose "
    "(seção 3 - Ponto médio)."
)
# Controle: trocamos o "ponto médio" por um ponto a um terço do caminho. Com essa troca,
# o segmento mede 2/3 de AB (e não metade), e a função deve devolver False.
ponto_medio_verdadeiro = ponto_medio
try:
    def ponto_medio(P, Q):
        return ponto_do_segmento(P, Q, 1 / 3)
    resultado_do_controle = testar_base_media((0, 0), (6, 0), (2, 4))
finally:
    ponto_medio = ponto_medio_verdadeiro
assert not resultado_do_controle, (
    "Tente de novo, revise: a função precisa calcular os pontos médios com ponto_medio e "
    "comparar com metade de AB; num triângulo de controle em que trocamos 'metade' por "
    "'um terço', ela deveria devolver False (seção 3 - Ponto médio)."
)
print("Certo!")

**(b) A prova analítica.** Agora prove a propriedade para **qualquer** triângulo. Escreva a prova na célula de texto abaixo (dê dois cliques nela para editar).

Sugestão: coloque $A(0, 0)$, $B(2b, 0)$ e $C(2c, 2d)$. *Por que usar $2b$, $2c$ e $2d$ em vez de $b$, $c$ e $d$? Pense no que acontece quando você calcula os pontos médios.*

*Escreva aqui a sua prova do item (b).*

In [ ]:
# Exercício Desafio 10: pontos de coordenadas inteiras. Quantos pontos com as duas
# coordenadas inteiras estão a uma distância EXATAMENTE 5 da origem?
# Use dois laços, um dentro do outro: percorra x de -5 a 5 e, para cada x, percorra
# y de -5 a 5. Guarde na lista pontos_ex10 os pontos (x, y) que servem e, em
# quantidade_ex10, quantos eles são.
pontos_ex10 = []
...  # complete aqui: os dois laços

quantidade_ex10 = ...

print(quantidade_ex10, pontos_ex10)

In [ ]:
# Verificação
assert quantidade_ex10 is not ... and quantidade_ex10 == 12, (
    "Tente de novo, revise: percorra x e y de −5 a 5 e compare x² + y² com 25 (sem raiz!) "
    "(seção 2.2 - De volta ao guarda do jogo)."
)
assert len(set(pontos_ex10)) == 12 and all(x**2 + y**2 == 25 for x, y in pontos_ex10), (
    "Tente de novo, revise: guarde em pontos_ex10 cada ponto (x, y) com x² + y² = 25, sem "
    "repetir (seção 2.2 - De volta ao guarda do jogo)."
)
print("Certo!")

Depois de acertar, rode a célula abaixo para ver os pontos que você encontrou.

In [ ]:
# Verificação visual do Exercício Desafio 10 (rode depois de acertar).
assert len(pontos_ex10) > 0, (
    "Tente de novo, revise: primeiro resolva o exercício acima "
    "(seção 2.2 - De volta ao guarda do jogo)."
)
fig = uga.criar_plano_cartesiano(titulo="Pontos inteiros a uma distância 5 da origem")
fig = uga.desenhar_circulo_de_alcance(fig, (0, 0), 5, cor=uga.CORES["apoio"], opacidade=0.05)
fig = uga.desenhar_ponto(fig, (0, 0), rotulo="O", cor=uga.CORES["apoio"])
fig = uga.desenhar_pontos(fig, pontos_ex10, cor=uga.CORES["destaque"])
uga.mostrar(fig)
print("📎 Esses pontos estão todos na circunferência de centro O e raio 5. No notebook 08")
print("você vai escrever a equação dela: x² + y² = 25. E repare no 3-4-5: um terno")
print("pitagórico (📎 0416a_triangulos_retangulos.ipynb, módulo 04).")

## Conexões

**Isso depende de:**
- [`00_introducao_python.ipynb`](../00_fundamentos_e_python/00_introducao_python.ipynb) (Python básico)
- [`0104a_relacoes.ipynb`](../01_conjuntos_e_funcoes/0104a_relacoes.ipynb) (par ordenado, plano cartesiano e relação inversa) e [`0110a_funcao_modular.ipynb`](../01_conjuntos_e_funcoes/0110a_funcao_modular.ipynb) (módulo)
- [`0402a_segmento_de_reta.ipynb`](../04_geometria_plana/0402a_segmento_de_reta.ipynb) (ponto médio e a fórmula da distância usada "de confiança")
- [`0404a_triangulos_conceito_elementos_e_classificacao.ipynb`](../04_geometria_plana/0404a_triangulos_conceito_elementos_e_classificacao.ipynb) (classificação de triângulos)
- [`0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb`](../04_geometria_plana/0408a_quadrilateros_definicao_elementos_e_trapezios.ipynb) (base média, no Desafio 9) e [`0409a_paralelogramos_e_casos_particulares.ipynb`](../04_geometria_plana/0409a_paralelogramos_e_casos_particulares.ipynb) (diagonais do paralelogramo, no Desafio 8)
- [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (baricentro e a razão 2 : 1)
- [`0414a_teorema_de_tales.ipynb`](../04_geometria_plana/0414a_teorema_de_tales.ipynb) (Teorema de Tales e divisão de um segmento numa razão)
- [`0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb`](../04_geometria_plana/0415a_semelhanca_de_triangulos_e_potencia_de_ponto.ipynb) (semelhança de triângulos)
- [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Pitágoras, recíproca e mediana relativa à hipotenusa)
- [`0617a_superficies_e_solidos_de_revolucao.ipynb`](../06_geometria_espacial/0617a_superficies_e_solidos_de_revolucao.ipynb) (centroides, no módulo 06)

**Isso será usado em:**
- 📌 Alinhamento e áreas por determinantes (notebook 02: como saber se três pontos estão alinhados)
- 📌 Vetores no plano (notebook 03: o deslocamento $(\Delta x, \Delta y)$ vai virar um vetor)
- 📌 Equações da reta (notebook 04: a fórmula do ponto do segmento, com qualquer $t$, vira a equação paramétrica da reta)
- 📌 Distância de ponto a reta e aplicações (notebook 06)
- 📌 Circunferência: equações (notebook 08: a condição $d^2 = r^2$ vira a equação da circunferência)
- 📌 Sistema de coordenadas no espaço (notebook 16: as mesmas fórmulas com uma coordenada a mais)

Para revisar os termos do módulo, use o [glossário de Geometria Analítica](glossario.md).